# 02 - ETL Pipeline: NASA NeoWs → Supabase

Pipeline completo de extracción, transformación y carga de datos de asteroides.

**Fuente:** NASA NeoWs API  
**Destino:** Supabase (PostgreSQL)  
**Período:** Enero 2024 → Abril 2026  
**Registros finales:** 11.775 asteroides únicos

## 1. Configuración e imports

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
from supabase import create_client
import os
import time

load_dotenv(dotenv_path='../.env')
API_KEY = os.getenv("NASA_API_KEY")

# Verificar que la key cargó bien
print("API Key cargada:", API_KEY is not None)

API Key cargada: True


## 2. Funciones de extracción y transformación

`get_asteroids()` — llama a la API por rango de fechas (máximo 7 días por request).  
`extraer_campos()` — normaliza cada asteroide y selecciona solo los campos relevantes para el modelo.

In [ ]:
def get_asteroids(start_date, end_date):
    url = "https://api.nasa.gov/neo/rest/v1/feed"
    params = {"start_date": start_date, "end_date": end_date, "api_key": API_KEY}
    response = requests.get(url, params=params)
    return response.json()['near_earth_objects']

def extraer_campos(asteroide):
    close_approach = asteroide['close_approach_data'][0]
    return {
        'id': asteroide['id'],
        'name': asteroide['name'],
        'absolute_magnitude_h': asteroide['absolute_magnitude_h'],
        'diameter_min_km': asteroide['estimated_diameter']['kilometers']['estimated_diameter_min'],
        'diameter_max_km': asteroide['estimated_diameter']['kilometers']['estimated_diameter_max'],
        'velocity_km_per_hour': float(close_approach['relative_velocity']['kilometers_per_hour']),
        'miss_distance_km': float(close_approach['miss_distance']['kilometers']),
        'close_approach_date': close_approach['close_approach_date'],
        'is_potentially_hazardous': asteroide['is_potentially_hazardous_asteroid']
    }

## 3. Descarga, deduplicación y carga

**Decisión de diseño:** Se conserva el primer acercamiento registrado por asteroide.
El campo `is_potentially_hazardous` es una propiedad fija del objeto, no del evento,
por lo que no se pierde información relevante para el modelo.

In [ ]:
load_dotenv(dotenv_path='../.env')

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

start = datetime(2024, 1, 1)
end = datetime(2026, 4, 1)
todos_los_asteroides = []
current = start

while current < end:
    siguiente = min(current + timedelta(days=7), end)
    datos = get_asteroids(current.strftime('%Y-%m-%d'), siguiente.strftime('%Y-%m-%d'))
    for fecha, asteroides in datos.items():
        for asteroide in asteroides:
            todos_los_asteroides.append(extraer_campos(asteroide))
    print(f"✓ {current.strftime('%Y-%m-%d')} — acumulados: {len(todos_los_asteroides)}")
    current = siguiente + timedelta(days=1)
    time.sleep(0.5)

df = pd.DataFrame(todos_los_asteroides)

# Cuántos registros teníamos
print("Antes de deduplicar:", len(df))

# Nos quedamos con el primer acercamiento de cada asteroide
df = df.drop_duplicates(subset='id', keep='first')

# Cuántos quedaron
print("Después de deduplicar:", len(df))

print("\nCargando en Supabase...")
batch_size = 500
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size].to_dict(orient='records')
    supabase.table('asteroids').upsert(batch).execute()
    print(f"  ✓ Insertados registros {i} a {i+len(batch)}")

print("\nListo")

✓ 2024-01-01 — acumulados: 124
✓ 2024-01-09 — acumulados: 280
✓ 2024-01-17 — acumulados: 420
✓ 2024-01-25 — acumulados: 572
✓ 2024-02-02 — acumulados: 732
✓ 2024-02-10 — acumulados: 887
✓ 2024-02-18 — acumulados: 1020
✓ 2024-02-26 — acumulados: 1185
✓ 2024-03-05 — acumulados: 1314
✓ 2024-03-13 — acumulados: 1441
✓ 2024-03-21 — acumulados: 1575
✓ 2024-03-29 — acumulados: 1729
✓ 2024-04-06 — acumulados: 1887
✓ 2024-04-14 — acumulados: 2068
✓ 2024-04-22 — acumulados: 2215
✓ 2024-04-30 — acumulados: 2356
✓ 2024-05-08 — acumulados: 2502
✓ 2024-05-16 — acumulados: 2614
✓ 2024-05-24 — acumulados: 2732
✓ 2024-06-01 — acumulados: 2861
✓ 2024-06-09 — acumulados: 2962
✓ 2024-06-17 — acumulados: 3081
✓ 2024-06-25 — acumulados: 3202
✓ 2024-07-03 — acumulados: 3316
✓ 2024-07-11 — acumulados: 3418
✓ 2024-07-19 — acumulados: 3525
✓ 2024-07-27 — acumulados: 3641
✓ 2024-08-04 — acumulados: 3770
✓ 2024-08-12 — acumulados: 3901
✓ 2024-08-20 — acumulados: 4039
✓ 2024-08-28 — acumulados: 4203
✓ 2024-09-05 —